# Phase 7b — Dense Search, Evaluation & Analysis  (v4.0)

This notebook picks up **after** Notebook 7a (v4.0) has:

1. Encoded the corpus into dense embeddings (`.npy` files in `models/`)
2. Inserted embeddings into ChromaDB (persistent on disk in `chroma_db/`)

## What's new in v4.0

- **Enriched text builder** — `build_doc_text` matches 7a v4.0 (country names,
  program labels, dataset sources, identifiers)
- **Richer search display** — results now show country and sanctions program
  alongside caption and schema
- **Both models loaded simultaneously** — `dense_search("query", model='minilm')`

## Pipeline

1. **Configure** — set `RUN_TAG` to match your 7a output, pick a default model
2. **Auto-detect** project paths
3. **Load both** ChromaDB collections and query encoders
4. **Search** with either model via `dense_search()`
5. **Export** pooling files, build qrels, evaluate


---
## 0 · Configuration

Set **two things**:
1. `RUN_TAG` — must match the tag from Notebook 7a (printed at the end of 7a)
2. `DEFAULT_MODEL` — which model `dense_search()` uses when you don't specify

Both models are always loaded. You can override per-call with `model='minilm'`
or `model='bge-m3'`.

| Key | HF Name | Dim | Notes |
|-----|---------|-----|-------|
| `minilm` | all-MiniLM-L6-v2 | 384 | Fast baseline |
| `bge-m3` | BAAI/bge-m3 | 1024 | Multilingual, long-context |

In [1]:
# ======================================================================
# CONFIGURATION — set these to match your Notebook 7a run
# ======================================================================

# The RUN_TAG from 7a (e.g. '50K_20260407', 'FULL_20260407')
# Check your models/ folder or 7a's final summary for the exact tag.
RUN_TAG = '77K_20260407'

# Default model for dense_search() — can always override per-call
#   DEFAULT_MODEL = 'minilm'     # all-MiniLM-L6-v2  (384-d, fast)
#   DEFAULT_MODEL = 'bge-m3'     # BAAI/bge-m3       (1024-d, multilingual)
DEFAULT_MODEL = 'bge-m3'

print(f"RUN_TAG       : {RUN_TAG}")
print(f"DEFAULT_MODEL : {DEFAULT_MODEL}")

RUN_TAG       : 77K_20260407
DEFAULT_MODEL : bge-m3


---
## 1 · Imports and Setup

In [3]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import time
import warnings
import torch
import psutil
import logging
from pathlib import Path
from collections import defaultdict

from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
import chromadb
import pycountry

from rank_bm25 import BM25Okapi
from ranx import Qrels, Run, evaluate, compare
from rapidfuzz import fuzz

import nltk
import spacy
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
STOP_WORDS = set(stopwords.words('english'))

warnings.filterwarnings('ignore')
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
sns.set_style('whitegrid')


def mem_report(tag=""):
    proc = psutil.Process()
    rss = proc.memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    print(f"  [MEM {tag}] RSS={rss:.2f} GB | Available={avail:.2f} GB")


mem_report("after imports")
print("All imports loaded.")


  [MEM after imports] RSS=0.97 GB | Available=8.86 GB
All imports loaded.


In [4]:
# ======================================================================
# Auto-detect project paths  (same logic as Notebook 7a)
# ======================================================================

def find_corpus():
    patterns = [
        'processed/full/documents.jsonl',
        'data/processed/full/documents.jsonl',
        'IR_project/processed/full/documents.jsonl',
    ]
    current = Path.cwd().resolve()
    for _ in range(6):
        for pat in patterns:
            candidate = current / pat
            if candidate.exists():
                return candidate.resolve()
        current = current.parent
    return None

CORPUS_PATH = find_corpus()
if CORPUS_PATH is None:
    raise FileNotFoundError(
        "Could not find 'documents.jsonl' in the project tree.\n"
        f"  Searched upward from: {Path.cwd()}\n"
        "  Set CORPUS_PATH manually if needed."
    )

PROJECT_ROOT = CORPUS_PATH.parent.parent.parent
MODELS_DIR   = PROJECT_ROOT / 'models'
CHROMA_DIR   = PROJECT_ROOT / 'chroma_db'

def find_path(*candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

QUERIES_PATH = find_path(PROJECT_ROOT / 'evaluation' / 'queries_part_a.xlsx',
                         PROJECT_ROOT.parent / 'data' / 'evaluation' / 'queries_part_a.xlsx',
                         '../data/evaluation/queries_part_a.xlsx')
QRELS_PATH   = find_path(PROJECT_ROOT / 'evaluation' / 'qrels.trec',
                          PROJECT_ROOT.parent / 'data' / 'evaluation' / 'qrels.trec',
                          '../data/evaluation/qrels.trec')
EVAL_OUT     = QUERIES_PATH.parent if QUERIES_PATH else PROJECT_ROOT

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected : {gpu_name}  ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA GPU — falling back to CPU.")

# ======================================================================
# Model Registry (must match Notebook 7a)
# ======================================================================
MODEL_REGISTRY = {
    'minilm': {
        'name':       'all-MiniLM-L6-v2',
        'dim':        384,
        'trust_remote_code': False,
        'notes':      'Lightweight baseline — fast, good quality',
    },
    'bge-m3': {
        'name':       'BAAI/bge-m3',
        'dim':        1024,
        'trust_remote_code': False,
        'notes':      'Multilingual M3 — dense + sparse + ColBERT',
    },
}

# ======================================================================
# Validate both models against 7a outputs using RUN_TAG
# ======================================================================
MODEL_CONFIGS = {}
_available = []
_missing   = []

for mk, cfg in MODEL_REGISTRY.items():
    suffix = mk.replace('-', '_')
    emb_path = MODELS_DIR / f'doc_embeddings_{suffix}_{RUN_TAG}.npy'
    ids_path = MODELS_DIR / f'doc_ids_{suffix}_{RUN_TAG}.json'
    coll_name = f'opensanctions_{suffix}_{RUN_TAG}'

    if emb_path.exists() and ids_path.exists():
        MODEL_CONFIGS[mk] = {
            **cfg,
            'key':               mk,
            'embedding_cache':   emb_path,
            'doc_ids_cache':     ids_path,
            'chroma_collection': coll_name,
        }
        _available.append(mk)
    else:
        _missing.append(mk)

if not MODEL_CONFIGS:
    raise FileNotFoundError(
        f"No embeddings found for RUN_TAG='{RUN_TAG}' in {MODELS_DIR}.\n"
        f"  Expected files like: doc_embeddings_minilm_{RUN_TAG}.npy\n"
        "  Run Notebook 7a first, then copy the RUN_TAG it prints."
    )

if DEFAULT_MODEL not in MODEL_CONFIGS:
    # Fall back to whatever is available
    DEFAULT_MODEL = _available[0]
    print(f"  Note: requested default not available, falling back to '{DEFAULT_MODEL}'")

print(f"Project root  : {PROJECT_ROOT}")
print(f"Models dir    : {MODELS_DIR}")
print(f"ChromaDB      : {CHROMA_DIR}")
print(f"Queries       : {QUERIES_PATH}")
print(f"Qrels         : {QRELS_PATH}")
print(f"Device        : {DEVICE}")
print(f"RUN_TAG       : {RUN_TAG}")
print()
print(f"Models available ({len(_available)}): {_available}")
if _missing:
    print(f"Models missing  ({len(_missing)}): {_missing}")
print(f"Default model   : {DEFAULT_MODEL}")
for mk in _available:
    mc = MODEL_CONFIGS[mk]
    tag = " <-- DEFAULT" if mk == DEFAULT_MODEL else ""
    print(f"  {mk:10s}  dim={mc['dim']}  col='{mc['chroma_collection']}'{tag}")

GPU detected : NVIDIA GeForce RTX 4080  (17.2 GB VRAM)
Project root  : C:\Users\marek\OneDrive\IR_2\IR_project
Models dir    : C:\Users\marek\OneDrive\IR_2\IR_project\models
ChromaDB      : C:\Users\marek\OneDrive\IR_2\IR_project\chroma_db
Queries       : None
Qrels         : None
Device        : cuda
RUN_TAG       : 77K_20260407

Models available (2): ['minilm', 'bge-m3']
Default model   : bge-m3
  minilm      dim=384  col='opensanctions_minilm_77K_20260407'
  bge-m3      dim=1024  col='opensanctions_bge_m3_77K_20260407' <-- DEFAULT


---
## 2 · Load Corpus and Queries

In [5]:
def load_corpus(path):
    if path is None:
        raise FileNotFoundError("CORPUS_PATH is None.")
    docs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                docs.append(json.loads(line))
    return docs

corpus = load_corpus(CORPUS_PATH)
doc_ids = [doc.get('doc_id', doc.get('id', f'doc_{i}')) for i, doc in enumerate(corpus)]
print(f"Corpus: {len(corpus):,} documents")

# ---- Queries ----
FALLBACK_QUERIES = [
    {"query_id": "Q01", "query_text": corpus[0].get('caption', 'sanction'), "query_type": "exact_name"},
    {"query_id": "Q02", "query_text": corpus[1].get('caption', 'person'),   "query_type": "exact_name"},
    {"query_id": "Q03", "query_text": "Russia sanctions",                   "query_type": "country"},
    {"query_id": "Q04", "query_text": "Iran nuclear program",               "query_type": "country"},
    {"query_id": "Q05", "query_text": "sanctioned company",                 "query_type": "entity_type"},
    {"query_id": "Q06", "query_text": "vessel ship maritime",               "query_type": "entity_type"},
    {"query_id": "Q07", "query_text": "bank financial institution",         "query_type": "entity_type"},
    {"query_id": "Q08", "query_text": "terrorism financing",                "query_type": "topic"},
    {"query_id": "Q09", "query_text": "arms dealer weapons",                "query_type": "topic"},
    {"query_id": "Q10", "query_text": "cryptocurrency wallet",              "query_type": "entity_type"},
]

if QUERIES_PATH is not None:
    queries_df = pd.read_excel(QUERIES_PATH)
    col_map = {}
    for col in queries_df.columns:
        lower = col.lower().strip()
        if 'id' in lower and 'query' in lower:   col_map[col] = 'query_id'
        elif lower in ('query', 'query_text', 'text'): col_map[col] = 'query_text'
        elif 'type' in lower:                     col_map[col] = 'query_type'
    queries_df = queries_df.rename(columns=col_map)
    if 'query_id'   not in queries_df.columns:
        queries_df['query_id'] = [f'Q{i+1:02d}' for i in range(len(queries_df))]
    if 'query_text' not in queries_df.columns:
        queries_df['query_text'] = queries_df.iloc[:, 0].astype(str)
    if 'query_type' not in queries_df.columns:
        queries_df['query_type'] = 'unknown'
    print(f"Queries: {len(queries_df)} from file")
else:
    queries_df = pd.DataFrame(FALLBACK_QUERIES)
    print(f"Queries: {len(queries_df)} (fallback)")

queries_df = queries_df.astype({'query_id': str, 'query_text': str, 'query_type': str})
mem_report("after corpus+queries load")

Corpus: 1,248,365 documents
Queries: 10 (fallback)
  [MEM after corpus+queries load] RSS=6.09 GB | Available=3.95 GB


---
## 3 · Helper Functions

In [6]:
def get_tokens(doc):
    tokens = doc.get('tokens')
    if tokens and isinstance(tokens, list):
        return tokens
    blob = doc.get('text_blob', '')
    return blob.lower().split() if blob else []


def preprocess_query(query_text):
    text = query_text.lower().strip()
    doc = nlp(text)
    return [
        token.lemma_.lower().strip()
        for token in doc
        if token.lemma_.lower().strip()
        and token.lemma_.isalpha()
        and token.lemma_.lower() not in STOP_WORDS
        and len(token.lemma_) > 1
    ]


# ======================================================================
# ISO-3166-1 alpha-2 → full country name
# ======================================================================

_COUNTRY_CACHE = {}

def _country_name(code):
    """Convert 2-letter ISO code to full country name, with cache."""
    code = code.strip().upper()
    if code in _COUNTRY_CACHE:
        return _COUNTRY_CACHE[code]
    try:
        name = pycountry.countries.get(alpha_2=code).name
    except (AttributeError, LookupError):
        _EXTRA = {
            'XK': 'Kosovo', 'TW': 'Taiwan', 'PS': 'Palestine',
            'AN': 'Netherlands Antilles', 'CS': 'Serbia and Montenegro',
            'SU': 'Soviet Union', 'YU': 'Yugoslavia', 'XX': 'Unknown',
        }
        name = _EXTRA.get(code, code)
    _COUNTRY_CACHE[code] = name
    return name


# ======================================================================
# Program-ID → human-readable label
# ======================================================================

_PROGRAM_LABELS = {
    'US-OFAC':     'U.S. OFAC sanctions',
    'US-NARCO':    'U.S. Foreign Narcotics Kingpin Designation Act',
    'US-GLOMAG':   'U.S. Global Magnitsky sanctions',
    'US-BIS':      'U.S. Bureau of Industry and Security',
    'US-BIS-DPL':  'U.S. BIS Denied Persons List',
    'US-BIS-EL':   'U.S. BIS Entity List',
    'US-SDGT':     'U.S. Specially Designated Global Terrorist',
    'US-SDN':      'U.S. Specially Designated Nationals',
    'US-SAM':      'U.S. SAM Exclusions',
    'US-HHS-OIG':  'U.S. HHS Office of Inspector General exclusions',
    'EU-UKR':      'EU Ukraine-related sanctions',
    'EU-FSF':      'EU Financial Sanctions',
    'UA-SA1644':   'Ukraine NSDC sanctions',
    'CA-SEMA':     'Canada Special Economic Measures Act',
    'INTERPOL-RN': 'Interpol Red Notice',
    'SECO':        'Swiss SECO sanctions',
    'AU-SANCTIONS': 'Australian sanctions',
    'GB-HMT':      'UK HM Treasury sanctions',
}

def _program_label(prog_id):
    pid = prog_id.strip().upper()
    if pid in _PROGRAM_LABELS:
        return _PROGRAM_LABELS[pid]
    for prefix in sorted(_PROGRAM_LABELS, key=len, reverse=True):
        if pid.startswith(prefix):
            return _PROGRAM_LABELS[prefix]
    return pid.replace('-', ' ').replace('_', ' ')


# ======================================================================
# Dataset slug → human-readable source label
# ======================================================================

_DATASET_LABELS = {
    'us_ofac_sdn':                 'U.S. OFAC SDN List',
    'us_ofac_cons':                'U.S. OFAC Consolidated List',
    'us_sam_exclusions':           'U.S. SAM Exclusions',
    'us_trade_csl':                'U.S. Consolidated Screening List',
    'us_bis_denied':               'U.S. BIS Denied Persons',
    'us_bis_entity':               'U.S. BIS Entity List',
    'eu_journal_sanctions':        'EU Official Journal sanctions',
    'eu_fsf':                      'EU Financial Sanctions',
    'gb_hmt_sanctions':            'UK HM Treasury sanctions',
    'un_sc_sanctions':             'UN Security Council sanctions',
    'interpol_red_notices':        'Interpol Red Notices',
    'ch_seco_sanctions':           'Swiss SECO sanctions',
    'opencorporates':              'OpenCorporates registry',
    'ext_us_ofac_press_releases':  'OFAC press releases',
    'ua_nsdc_sanctions':           'Ukraine NSDC sanctions',
    'ca_sema_sanctions':           'Canada SEMA sanctions',
    'au_dfat_sanctions':           'Australian DFAT sanctions',
}

def _dataset_label(ds):
    return _DATASET_LABELS.get(ds, ds.replace('_', ' ').title())


# ======================================================================
# build_doc_text v2 (matches 7a v4.0)
# ======================================================================

def build_doc_text(doc):
    """
    Build a natural-language text representation of an entity for embedding.
    v4.0: reads metadata (country, programId, datasets) and identifiers.
    """
    parts = []

    caption = doc.get('caption', '')
    schema  = doc.get('schema', '')
    if caption:
        parts.append(f"{caption}.")
    if schema:
        parts.append(f"Type: {schema}.")

    meta = doc.get('metadata', {})

    countries_raw = meta.get('country', [])
    if countries_raw and isinstance(countries_raw, list):
        country_names = [_country_name(c) for c in countries_raw[:5]]
        parts.append(f"Country: {', '.join(country_names)}.")

    programs_raw = meta.get('programId', [])
    if programs_raw and isinstance(programs_raw, list):
        seen = set()
        prog_labels = []
        for p in programs_raw[:4]:
            lbl = _program_label(p)
            if lbl not in seen:
                prog_labels.append(lbl)
                seen.add(lbl)
        if prog_labels:
            parts.append(f"Sanctioned under: {'; '.join(prog_labels)}.")

    datasets_raw = meta.get('datasets', [])
    if datasets_raw and isinstance(datasets_raw, list):
        ds_labels = []
        seen_ds = set()
        for ds in datasets_raw[:6]:
            lbl = _dataset_label(ds)
            if lbl not in seen_ds:
                ds_labels.append(lbl)
                seen_ds.add(lbl)
        if ds_labels:
            parts.append(f"Listed in: {'; '.join(ds_labels[:3])}.")

    idents = doc.get('identifiers', {})
    if isinstance(idents, dict):
        id_parts = []
        if idents.get('imoNumber'):
            imo_vals = idents['imoNumber']
            if isinstance(imo_vals, list):
                id_parts.append(f"IMO number: {', '.join(str(v) for v in imo_vals[:2])}")
        if idents.get('callSign'):
            cs_vals = idents['callSign']
            if isinstance(cs_vals, list):
                id_parts.append(f"Call sign: {', '.join(str(v) for v in cs_vals[:2])}")
        if idents.get('registrationNumber'):
            reg_vals = idents['registrationNumber']
            if isinstance(reg_vals, list):
                id_parts.append(f"Registration: {', '.join(str(v) for v in reg_vals[:2])}")
        if idents.get('innCode'):
            inn_vals = idents['innCode']
            if isinstance(inn_vals, list):
                id_parts.append(f"INN code: {', '.join(str(v) for v in inn_vals[:2])}")
        if id_parts:
            parts.append(' '.join(id_parts) + '.')

    if len(parts) <= 2:
        blob = doc.get('text_blob', '')
        if blob:
            parts.append(blob[:500])

    return ' '.join(parts)


print("Helpers defined (v4.0 — enriched build_doc_text).")



Helpers defined (v4.0 — enriched build_doc_text).


---
## 4 · Connect to ChromaDB & Load Query Models

Both models are loaded simultaneously so you can switch between them
in any `dense_search()` call without restarting.

In [11]:
# ---- Connect to ChromaDB collections for ALL available models ----
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

available_collections = [
    c.name if hasattr(c, 'name') else str(c)
    for c in chroma_client.list_collections()
]
print(f"ChromaDB directory  : {CHROMA_DIR}")
print(f"Collections on disk : {available_collections}")
print()

model_collections = {}
for mk in MODEL_CONFIGS:
    mc = MODEL_CONFIGS[mk]
    coll_name = mc['chroma_collection']
    if coll_name not in available_collections:
        print(f"  WARNING: '{coll_name}' not in ChromaDB — skipping '{mk}'.")
        continue
    coll = chroma_client.get_collection(name=coll_name)
    model_collections[mk] = coll
    print(f"  Connected: {mk:10s} -> '{coll_name}' ({coll.count():,} docs)")

if not model_collections:
    raise RuntimeError(
        "No ChromaDB collections found. Run Notebook 7a first."
    )

# ---- Load query encoders for ALL connected models ----
print()
query_models = {}
for mk in model_collections:
    mc = MODEL_CONFIGS[mk]
    print(f"Loading query encoder: {mk} -> {mc['name']} ...", end=" ")

    if DEVICE == 'cuda':
        # Load to CPU first, convert to fp16, then move to GPU
        # Avoids the fp32 peak that causes OOM with both models loaded
        _m = SentenceTransformer(mc['name'], device='cpu')
        _m = _m.half().to(DEVICE)
        print("done (fp16, direct)")
    else:
        _m = SentenceTransformer(mc['name'], device=DEVICE)
        print("done (cpu)")

    query_models[mk] = _m

    # Free any leftover allocation from the load
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f"\nModels ready for search: {list(query_models.keys())}")
print(f"Default model          : {DEFAULT_MODEL}")
mem_report("after all models loaded")


ChromaDB directory  : C:\Users\marek\OneDrive\IR_2\IR_project\chroma_db
Collections on disk : ['opensanctions_minilm_23K_20260407', 'opensanctions_bge_m3_23K_20260407', 'opensanctions_bge_m3_16K_20260407', 'opensanctions_minilm_16K_20260407', 'opensanctions_bge_m3', 'opensanctions_minilm_77K_20260407', 'opensanctions_bge_m3_77K_20260407', 'opensanctions_minilm']

  Connected: minilm     -> 'opensanctions_minilm_77K_20260407' (77,000 docs)
  Connected: bge-m3     -> 'opensanctions_bge_m3_77K_20260407' (77,000 docs)

Loading query encoder: minilm -> all-MiniLM-L6-v2 ... 

done (fp16, direct)
Loading query encoder: bge-m3 -> BAAI/bge-m3 ... done (fp16, direct)

Models ready for search: ['minilm', 'bge-m3']
Default model          : bge-m3
  [MEM after all models loaded] RSS=7.40 GB | Available=4.03 GB


---
## 5 · Dense Search

A single function that can query **either** model. The `model`
parameter selects which encoder + collection to use.

```python
# Uses DEFAULT_MODEL:
dense_search("arms trafficker")

# Explicitly pick a model:
dense_search("arms trafficker", model='minilm')
dense_search("arms trafficker", model='bge-m3')

# Compare both side-by-side:
dense_search("arms trafficker", model='minilm')
dense_search("arms trafficker", model='bge-m3')
```

In [13]:
def dense_search(query_text, top_k=20, model=None, verbose=True):
    """
    Search ChromaDB by semantic similarity.

    Parameters
    ----------
    query_text : str       — natural language query
    top_k      : int       — number of results to return
    model      : str       — 'minilm' or 'bge-m3' (None = DEFAULT_MODEL)
    verbose    : bool      — print results to stdout

    Returns
    -------
    dict — raw ChromaDB query results (keys: ids, distances, metadatas)
    """
    mk = model or DEFAULT_MODEL

    if mk not in query_models:
        raise ValueError(
            f"Model '{mk}' not loaded. Available: {list(query_models.keys())}"
        )

    qm   = query_models[mk]
    coll = model_collections[mk]

    q_emb = qm.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

    results = coll.query(
        query_embeddings=q_emb,
        n_results=top_k,
        include=['distances', 'metadatas'],
    )

    if verbose:
        print(f"{'='*80}")
        print(f"[{mk}] Query: '{query_text}'")
        print(f"{'-'*80}")
        for rank, (did, dist, meta) in enumerate(
            zip(results['ids'][0], results['distances'][0], results['metadatas'][0]), 1
        ):
            similarity = 1.0 - dist
            caption = meta.get('caption', 'N/A')
            schema  = meta.get('schema', '')
            country = meta.get('country', '')
            program = meta.get('program', '')
            # Build a compact info line
            info = f"({schema})"
            if country:
                info += f"  [{country}]"
            if program:
                # Show first program only for compact display
                first_prog = program.split(',')[0].strip()
                info += f"  {first_prog}"
            print(f"  #{rank:2d}  [sim={similarity:.4f}]  {did}  {caption}  {info}")
        print()

    return results



### 5.1 · Sanity Checks

In [15]:
# ---- Run sanity checks with BOTH models ----
for mk in query_models:
    print(f"{'='*60}")
    print(f"  Sanity checks: {mk}")
    print(f"{'='*60}\n")
    _ = dense_search(corpus[0].get('caption', 'sanctions'), model=mk)
    _ = dense_search("arms trafficker weapons smuggling", model=mk)
    _ = dense_search("North Korean military officials", model=mk)
    _ = dense_search("Leader of Russia", model=mk)

  Sanity checks: minilm

[minilm] Query: 'Myanmar Yatai International Holding Group Co., LTD.'
--------------------------------------------------------------------------------
  # 1  [sim=0.7805]  NK-223CQDBzp8MRkdJMDiqXn3  Myanmar Yatai International Holding Group Co., LTD.  (Company)  [MM]  US-GLOMAG
  # 2  [sim=0.6360]  NK-eMctTHz2VCPE4uPvCLmmJQ  Xinjiang Yatai Textile Co., Ltd.  (Organization)  [CN]
  # 3  [sim=0.6253]  NK-4eLyFAtcwGFid5L42EaaAb  Kunlun Holding Company Ltd.  (Company)  [CN, VG]  US-IRAN
  # 4  [sim=0.6158]  NK-EsJoq3nyAYvFhn6wVw222T  Kashi Yayi Clothing Co., Ltd  (Organization)  [CN]
  # 5  [sim=0.6114]  NK-VDQgVTg32DgdhU8wouK9t9  EXCELLENCE MINERAL MANUFACTURING CO., LTD.  (Company)  [MM]
  # 6  [sim=0.6028]  NK-XVtACvFUMohpPYP5wX6veF  Dynasty International Company Limited  (Company)  [MM]  CA-SEMA
  # 7  [sim=0.5998]  NK-NtKKeLKhBQqgMP5kSXajd9  Star Sapphire Group Pte. Ltd.  (Company)  [SG]  US-BURMA
  # 8  [sim=0.5997]  NK-CXfxTGAY8BN88U7q9iv9fJ  Yining Nantai S

In [17]:
# Side-by-side comparison on a single query
print("=" * 60)
print("  SIDE-BY-SIDE: 'Mexican cartel'")
print("=" * 60)
for mk in query_models:
    _ = dense_search("U.S. adventure travel operator advertising extended trips to Persia", 20, model=mk)

  SIDE-BY-SIDE: 'Mexican cartel'
[minilm] Query: 'U.S. adventure travel operator advertising extended trips to Persia'
--------------------------------------------------------------------------------
  # 1  [sim=0.4582]  NK-8Ju3tGATdqy49FVpMB4RpY  Flight Travel LLC  (Company)  [AM]  US-IRAN
  # 2  [sim=0.4316]  NK-9zVbQ8uwyELbRLHB9vx8j2  ENDEAVOR TRAVEL, L.L.C.  (Company)  [US]
  # 3  [sim=0.3954]  NK-JbZxXpqKt4iAgXe2fDozBi  Journey Investment Company  (Company)  [MH]  US-IRAN
  # 4  [sim=0.3890]  NK-cXtjfDD6EqVjnEu9VZLmti  ZARIN PERSIA INVESTMENT  (Organization)  [IR]  US-IRAN
  # 5  [sim=0.3869]  NK-3tGHSN4Q58f7aomq3p6ixC  SUN GROUP AIR TRAVEL AND AIR CARGO AND AIRPORT SERVICES LTD  (LegalEntity)  [IR]
  # 6  [sim=0.3859]  NK-Z8pzUYg39knUVwbSF6KEjY  GRIFFIN TRANSPORTATION, INC.  (Company)  [US]
  # 7  [sim=0.3842]  NK-HZnPZZ6eFncDipt62tdQqu  STS TRANSPORTATION, LLC  (Company)  [US]
  # 8  [sim=0.3841]  NK-TzhYN2gNdWA49TrxBZMVfX  GLOBAL TRANSPORTATION LEADERS INC.  (Company)  [US]
  #

---
## 6 · Load Query Files (Type 3 & Type 4)

Loads `queries_type_3.json` and `queries_type_4.json` from the project's
`data/queries/` folder. Each JSON has a `queries` array with fields:
`query_id`, `query_type`, `query_texts` (list — we take the first),
and `notes`.


In [19]:
# ======================================================================
# Auto-detect queries folder and load Type 3 + Type 4 query files
# ======================================================================

def find_queries_dir():
    """Search for data/queries/ folder in common project layouts."""
    candidates = [
        PROJECT_ROOT / 'data' / 'queries',
        PROJECT_ROOT.parent / 'data' / 'queries',
        PROJECT_ROOT / 'queries',
        PROJECT_ROOT / 'evaluation' / 'queries',
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    return None

QUERIES_DIR = find_queries_dir()

if QUERIES_DIR is None:
    raise FileNotFoundError(
        f"Could not find 'data/queries/' folder.\n"
        f"  Searched from: {PROJECT_ROOT}\n"
        f"  Set QUERIES_DIR manually if your layout differs:\n"
        f"    QUERIES_DIR = Path(r'C:\\...\\data\\queries')"
    )

print(f"Queries folder: {QUERIES_DIR}")
print(f"Files found:")
for f in sorted(QUERIES_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

# ---- Load Type 3 and Type 4 ----
def load_query_json(filepath):
    """Load a query JSON file and return a list of (query_id, query_type, query_text, notes) tuples."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    queries = data.get('queries', [])
    rows = []
    for q in queries:
        qid   = q.get('query_id', '')
        qtype = q.get('query_type', '')
        # query_texts is a list — take the first entry
        qtexts = q.get('query_texts', [])
        qtext  = qtexts[0] if qtexts else ''
        notes  = q.get('notes', '')
        rows.append({
            'query_id':    str(qid),
            'query_type':  str(qtype),
            'query_text':  str(qtext),
            'query_notes': str(notes),
        })
    return rows

pooling_queries = []

for fname in ['queries_type_3.json', 'queries_type_4.json']:
    fpath = QUERIES_DIR / fname
    if fpath.exists():
        rows = load_query_json(fpath)
        pooling_queries.extend(rows)
        print(f"\n  Loaded {fname}: {len(rows)} queries")
        for r in rows[:3]:
            print(f"    {r['query_id']}: {r['query_text'][:80]}...")
    else:
        print(f"\n  WARNING: {fname} not found in {QUERIES_DIR}")

pooling_queries_df = pd.DataFrame(pooling_queries)
print(f"\nTotal queries for pooling: {len(pooling_queries_df)}")
pooling_queries_df.head()


Queries folder: C:\Users\marek\OneDrive\IR_2\IR_project\data\queries
Files found:
  .gitkeep  (0 bytes)
  queries_type_1.json  (14,322 bytes)
  queries_type_2.json  (18,612 bytes)
  queries_type_3.json  (5,337 bytes)
  queries_type_4.json  (4,172 bytes)
  queries_type_5.json  (24,137 bytes)
  queries_type_6.json  (15,848 bytes)

  Loaded queries_type_3.json: 18 queries
    Q3_001: Russian oil tankers in the shadow fleet sanctioned for circumventing the price c...
    Q3_002: North Korean state-sponsored hacking group that stole cryptocurrency from online...
    Q3_003: UAE-based shipping management companies running previously Russian-operated tank...

  Loaded queries_type_4.json: 14 queries
    Q4_001: What vessels does Sovcomflot own or manage?...
    Q4_002: What entities are connected to the Lazarus Group through front companies or alia...
    Q4_003: What is the corporate structure around Sovcomflot including subsidiaries, managi...

Total queries for pooling: 32


,query_id,query_type,query_text,query_notes
0,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT
1,Q3_002,3,North Korean state-sponsored hacking group tha...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT
2,Q3_003,3,UAE-based shipping management companies runnin...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT
3,Q3_004,3,Sanctioned tankers that have been detained by ...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT
4,Q3_005,3,Russian state shipping company that is the lar...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT


---
## 7 · Generate Pooling File (Both Models, All Queries)

For each query, runs `dense_search()` with **both** models, then merges
the results into a single table where each unique `(query_id, doc_id)`
pair has columns for both models' ranks and scores.

Output columns:
```
query_id | query_type | query_text | query_notes |
doc_id | minilm_rank | minilm_score | bge-m3_rank | bge-m3_score |
in_minilm | in_bge-m3 | relevant | judge_notes |
caption | schema | country | program | text_preview
```

The `relevant` and `judge_notes` columns are left blank for human review.


In [21]:
# ======================================================================
# Configuration
# ======================================================================
TOP_K = 20   # results per model per query

# Build corpus lookup for text_preview
corpus_lookup = {
    doc.get('doc_id', doc.get('id', f'doc_{i}')): doc
    for i, doc in enumerate(corpus)
}

# ======================================================================
# Run all queries through both models, merge into pooling rows
# ======================================================================

all_pooling_rows = []

for q_idx, q_row in pooling_queries_df.iterrows():
    qid    = q_row['query_id']
    qtype  = q_row['query_type']
    qtext  = q_row['query_text']
    qnotes = q_row['query_notes']
    
    print(f"[{q_idx+1}/{len(pooling_queries_df)}] {qid}: {qtext[:70]}...")
    
    # ---- Collect results from each model ----
    model_results = {}
    for mk in query_models:
        results = dense_search(qtext, top_k=TOP_K, model=mk, verbose=False)
        # Build a dict: doc_id -> (rank, similarity, metadata)
        hits = {}
        for rank, (did, dist, meta) in enumerate(
            zip(results['ids'][0], results['distances'][0], results['metadatas'][0]), 1
        ):
            similarity = round(1.0 - dist, 9)
            hits[did] = {
                'rank':  rank,
                'score': similarity,
                'meta':  meta,
            }
        model_results[mk] = hits
    
    # ---- Merge: union of all doc_ids across both models ----
    all_doc_ids = set()
    for mk in model_results:
        all_doc_ids.update(model_results[mk].keys())
    
    # Sort by average rank (docs found by both models first, then by rank)
    def sort_key(did):
        ranks = []
        for mk in query_models:
            if did in model_results[mk]:
                ranks.append(model_results[mk][did]['rank'])
            else:
                ranks.append(TOP_K + 100)  # push missing to end
        return (len(query_models) - len([r for r in ranks if r <= TOP_K]), sum(ranks) / len(ranks))
    
    sorted_doc_ids = sorted(all_doc_ids, key=sort_key)
    
    for did in sorted_doc_ids:
        # Get metadata from whichever model found it
        meta = {}
        for mk in query_models:
            if did in model_results[mk]:
                meta = model_results[mk][did]['meta']
                break
        
        # Get text_preview from corpus
        corp_doc = corpus_lookup.get(did, {})
        text_blob = corp_doc.get('text_blob', '')
        if not text_blob:
            tokens = corp_doc.get('tokens', [])
            text_blob = ' '.join(tokens) if tokens else ''
        text_preview = text_blob[:300].strip() if text_blob else ''
        
        # Build the row
        minilm_hit = model_results.get('minilm', {}).get(did)
        bgem3_hit  = model_results.get('bge-m3', {}).get(did)
        
        row = {
            'query_id':     qid,
            'query_type':   qtype,
            'query_text':   qtext,
            'query_notes':  qnotes,
            'doc_id':       did,
            'minilm_rank':  minilm_hit['rank']  if minilm_hit else '',
            'minilm_score': minilm_hit['score'] if minilm_hit else '',
            'bge-m3_rank':  bgem3_hit['rank']   if bgem3_hit  else '',
            'bge-m3_score': bgem3_hit['score']  if bgem3_hit  else '',
            'in_minilm':    1 if minilm_hit else 0,
            'in_bge-m3':    1 if bgem3_hit  else 0,
            'relevant':     '',
            'judge_notes':  '',
            'caption':      meta.get('caption', ''),
            'schema':       meta.get('schema', ''),
            'country':      meta.get('country', ''),
            'program':      meta.get('program', ''),
            'text_preview': text_preview,
        }
        all_pooling_rows.append(row)

df_pooling = pd.DataFrame(all_pooling_rows)
print(f"\nPooling table: {len(df_pooling)} rows  ({len(pooling_queries_df)} queries × up to {TOP_K} docs × 2 models, deduplicated)")
print(f"Unique queries: {df_pooling['query_id'].nunique()}")
print(f"Unique docs:    {df_pooling['doc_id'].nunique()}")
df_pooling.head(10)


[1/32] Q3_001: Russian oil tankers in the shadow fleet sanctioned for circumventing t...
[2/32] Q3_002: North Korean state-sponsored hacking group that stole cryptocurrency f...
[3/32] Q3_003: UAE-based shipping management companies running previously Russian-ope...
[4/32] Q3_004: Sanctioned tankers that have been detained by port authorities for saf...
[5/32] Q3_005: Russian state shipping company that is the largest transporter of crud...
[6/32] Q3_006: Russian private military company responsible for deploying mercenaries...
[7/32] Q3_007: Russian state energy company's oil division sanctioned for exploration...
[8/32] Q3_008: Sanctioned individual who led Wagner Group operations in Sudan and was...
[9/32] Q3_009: Sanctioned subsidiary of a Russian energy conglomerate operating in Ky...
[10/32] Q3_010: Russian paramilitary sabotage unit linked to a larger private military...
[11/32] Q3_011: Sanctioned Russian microelectronics manufacturer whose chips are used ...
[12/32] Q3_012: Ira

,query_id,query_type,query_text,query_notes,doc_id,minilm_rank,minilm_score,bge-m3_rank,bge-m3_score,in_minilm,in_bge-m3,relevant,judge_notes,caption,schema,country,program,text_preview
0,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-UzJJRynpn9DTCFWBZPkvg4,1,0.505446,,,1,0,,,Volga Shipping Company ‘Victoria’ LLC,Company,RU,,volga shipping company victoria llc poi гур мо...
1,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-RUQd8v6Efqqyu2exY74VTQ,,,1,0.565221,0,1,,,Onyx,Vessel,RU,"US-RUSHAR, UA-WS-MARE, EU-MARE, CA-SEMA, SECO-...",onyx onyx high nefeli crown iі nefeli own iі p...
2,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-dUqfiSx25V84BZnZ6FUUfb,,,2,0.560141,0,1,,,Fregat OOO,Company,RU,"US-RUSHAR, SECO-UKRAINE, CA-SEMA, UA-SA1644",limited liability company fregat товариство об...
3,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-HkdNsACJr2Pe73in5Kdf8c,2,0.496822,,,1,0,,,CLOSED JOINT-STOCK COMPANY VOLGAERO,Company,RU,UA-WS-MILIND,cjsc volgaero closed joint stock company volga...
4,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-fAsV7vqAvsLK8cDXZr23xF,,,3,0.559355,0,1,,,EASTERN GLORY,Vessel,RU,"UA-WS-MARE, EU-MARE, CA-SEMA, AU-VESSELS, SECO...",eastern glory night glory night glory eastern ...
5,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-Xoqsj6fHaCKyHCKvwidsNk,3,0.491393,,,1,0,,,VOLGA SHIPPING COMPANY,Company,RU,"EU-UKR, SECO-UKRAINE, GB-RUS, UA-SA1644",акционерное общество судоходная компания волжс...
6,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-U8LWviLum93XKgs9XXtdPb,,,4,0.557722,0,1,,,NOCHNYE VOLKI,Company,RU,"UA-SA1644, US-UKRRUS-REL, CA-SEMA, EU-UKR, SEC...",nochniye volki natulvene mc molodezhnaya avton...
7,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-bmyhQDMYHd9gFeHNYppM74,4,0.490741,,,1,0,,,Middle Volga Shipping Co,Company,RU,,middle volga shipping co poi гур мо украіни ua...
8,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-TDKsmX54JtvVQM5khuN6aD,,,5,0.555501,0,1,,,Everest Energy,Vessel,RU,"US-RUSHAR, UA-WS-MARE, CA-SEMA, EU-MARE, AU-VE...",everest energy metagas everest everest energy ...
9,Q3_001,3,Russian oil tankers in the shadow fleet sancti...,SHADOW FLEET / DPRK CYBER / UAE SHELL MGMT,NK-eHiUxg5pke7ztNg7z9a9gL,5,0.470108,,,1,0,,,Volga-don Shipping JSC (current Company Azovo/...,Company,RU,,volga don shipping jsc current company azovo d...


---
## 8 · Export to Excel

Saves the pooling table as an Excel file in the evaluation/output folder.
The `relevant` and `judge_notes` columns are blank — ready for expert review.


In [23]:
# ======================================================================
# Export pooling to Excel
# ======================================================================

# Output directory — same as evaluation folder, or project root
POOLING_OUT = find_path(
    PROJECT_ROOT / 'data' / 'evaluation',
    PROJECT_ROOT / 'evaluation',
    PROJECT_ROOT / 'data',
)
if POOLING_OUT is None:
    POOLING_OUT = PROJECT_ROOT
    
POOLING_OUT.mkdir(parents=True, exist_ok=True)

out_filename = f'pooling_dense_type3_type4_{RUN_TAG}.xlsx'
out_path = POOLING_OUT / out_filename

# ---- Column order (matches your specification) ----
col_order = [
    'query_id', 'query_type', 'query_text', 'query_notes',
    'doc_id',
    'minilm_rank', 'minilm_score',
    'bge-m3_rank', 'bge-m3_score',
    'in_minilm', 'in_bge-m3',
    'relevant', 'judge_notes',
    'caption', 'schema', 'country', 'program', 'text_preview',
]

# Ensure all columns exist (in case a model wasn't loaded)
for col in col_order:
    if col not in df_pooling.columns:
        df_pooling[col] = ''

df_export = df_pooling[col_order].copy()

# ---- Write with formatting ----
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    df_export.to_excel(writer, index=False, sheet_name='pooling')
    
    # Auto-adjust column widths
    ws = writer.sheets['pooling']
    for col_idx, col_name in enumerate(col_order, 1):
        # Set reasonable widths based on content type
        if col_name == 'query_text':
            width = 60
        elif col_name == 'text_preview':
            width = 80
        elif col_name in ('query_notes', 'judge_notes', 'caption', 'program'):
            width = 35
        elif col_name in ('doc_id',):
            width = 28
        elif col_name in ('query_id', 'query_type', 'schema', 'country'):
            width = 14
        elif 'rank' in col_name:
            width = 12
        elif 'score' in col_name:
            width = 14
        elif col_name.startswith('in_'):
            width = 10
        elif col_name == 'relevant':
            width = 10
        else:
            width = 15
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = width

print(f"Pooling exported to:")
print(f"  {out_path}")
print(f"  {len(df_export)} rows, {df_export['query_id'].nunique()} queries")
print(f"\nColumns: {list(df_export.columns)}")
print(f"\nBreakdown by query type:")
print(df_export.groupby('query_type')['query_id'].nunique().to_string())
print(f"\nReady for expert review — fill in 'relevant' and 'judge_notes' columns.")


Pooling exported to:
  C:\Users\marek\OneDrive\IR_2\IR_project\data\pooling_dense_type3_type4_77K_20260407.xlsx
  1240 rows, 32 queries

Columns: ['query_id', 'query_type', 'query_text', 'query_notes', 'doc_id', 'minilm_rank', 'minilm_score', 'bge-m3_rank', 'bge-m3_score', 'in_minilm', 'in_bge-m3', 'relevant', 'judge_notes', 'caption', 'schema', 'country', 'program', 'text_preview']

Breakdown by query type:
query_type
3    18
4    14

Ready for expert review — fill in 'relevant' and 'judge_notes' columns.


---
## 9 · Build / Load Relevance Judgments

After expert review of the pooling file, the `relevant` column can be
imported back as qrels for formal evaluation with ranx.


In [ ]:
def generate_pseudo_qrels(corpus, queries_df, fuzzy_threshold=75, min_overlap=0.5):
    qrels_dict = defaultdict(dict)
    doc_names, doc_token_set = {}, {}
    for doc in corpus:
        did = doc.get('id', '')
        names = []
        caption = doc.get('caption', '')
        if caption:
            names.append(caption)
        props = doc.get('properties', {})
        if isinstance(props, dict):
            for f in ['name', 'alias', 'previousName', 'weakAlias']:
                vals = props.get(f, [])
                if isinstance(vals, list):
                    names.extend([str(v) for v in vals if v])
        doc_names[did] = names
        doc_token_set[did] = set(get_tokens(doc))

    for _, row in tqdm(list(queries_df.iterrows()), desc="Pseudo-qrels", unit="query"):
        qid   = str(row['query_id'])
        qtext = str(row['query_text'])
        qtokens_set = set(preprocess_query(qtext))
        for doc in corpus:
            did, rel = doc.get('id', ''), 0
            for name in doc_names.get(did, []):
                ratio = fuzz.token_sort_ratio(qtext.lower(), name.lower())
                if   ratio >= 95:               rel = max(rel, 2)
                elif ratio >= fuzzy_threshold:   rel = max(rel, 1)
            if qtokens_set and rel < 1:
                overlap = len(qtokens_set & doc_token_set.get(did, set()))
                if overlap / len(qtokens_set) >= min_overlap:
                    rel = max(rel, 1)
            if rel > 0:
                qrels_dict[qid][did] = rel
    return dict(qrels_dict)


qrels_dict = None
if QRELS_PATH and QRELS_PATH.exists():
    print(f"Loading qrels from {QRELS_PATH}...")
    qrels_dict = defaultdict(dict)
    with open(QRELS_PATH, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 4 and int(parts[3]) > 0:
                qrels_dict[parts[0]][parts[2]] = int(parts[3])
    qrels_dict = dict(qrels_dict)

if not qrels_dict or len(qrels_dict) == 0:
    print("Generating pseudo-qrels...")
    qrels_dict = generate_pseudo_qrels(corpus, queries_df)

valid_qrels = {q: r for q, r in qrels_dict.items() if len(r) > 0}
print(f"Evaluable queries: {len(valid_qrels)}")

---
## 10 · Evaluation

Use `dense_search()` with either model in your evaluation loops:

```python
# Run all queries through both models
for mk in query_models:
    for _, row in queries_df.iterrows():
        results = dense_search(row['query_text'], model=mk, verbose=False)
        # ... build Run object for ranx ...
```

Both models are loaded and ready — no restarts needed.
